# Experimento de Ingenieria de Datos: Reglas de Asociacion

Dataset: **Social Media User Activity Dataset**

Objetivo: encontrar reglas de asociacion interpretables entre comportamiento en redes sociales, estilo de vida y niveles de `self_reported_happiness`.

Este notebook usa el modulo `src/social_media_activity_pipeline.py` para mantener el flujo reutilizable y facil de ajustar.

## 1. Instalacion de dependencias

En Google Colab se instalaran `kaggle` para descargar el dataset y `mlxtend` para usar FP-Growth y reglas de asociacion.

In [ ]:
!pip install -q kaggle mlxtend

## 2. Importaciones y conexion con el modulo del proyecto

Si estas ejecutando este notebook desde el repositorio clonado en Colab, la ruta `../src` deberia funcionar. Si subes el archivo manualmente, ajusta `PROJECT_SRC`.

In [ ]:
from pathlib import Path
import os
import sys
import warnings
import zipfile

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

PROJECT_SRC = Path("../src")
if PROJECT_SRC.exists():
    sys.path.append(str(PROJECT_SRC.resolve()))
else:
    sys.path.append("/content/src")

from social_media_activity_pipeline import (
    TARGET_COLUMN,
    clean_selected_data,
    create_binary_matrix,
    default_analysis_columns,
    discretize_numeric_columns,
    filter_happiness_rules,
    generate_rules,
    load_dataset,
    mine_frequent_itemsets,
    reduce_rare_categories,
    report_duplicates,
    report_missing_values,
    rules_to_readable,
    sample_dataframe,
    split_column_types,
    summarize_columns,
    validate_binary_matrix,
)

## 3. Parametros editables

Estos valores se pueden ajustar durante el experimento. Para pruebas rapidas se recomienda usar muestra.

In [ ]:
TARGET_COLUMN = "self_reported_happiness"
MAX_BINS = 10
USE_SAMPLE = True
SAMPLE_SIZE = 100_000
RANDOM_STATE = 42

MIN_SUPPORT = 0.01
MIN_CONFIDENCE = 0.40
MIN_LIFT = 1.00

DATASET_SLUG = "sadiajavedd/social-media-user-activity-dataset"
DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

## 4. Descarga o carga del dataset

Opcion recomendada en Colab:

1. Crear un API token en Kaggle.
2. Subir el archivo `kaggle.json` cuando Colab lo solicite.
3. Ejecutar la descarga.

Si ya tienes el CSV, puedes subirlo manualmente y definir `CSV_PATH` al archivo correspondiente.

In [ ]:
# Ejecuta esta celda en Colab si quieres descargar desde Kaggle.
# Si ya tienes el CSV cargado, puedes saltarla y definir CSV_PATH manualmente.

from google.colab import files

if not Path("/root/.kaggle/kaggle.json").exists():
    uploaded = files.upload()
    if "kaggle.json" in uploaded:
        Path("/root/.kaggle").mkdir(parents=True, exist_ok=True)
        Path("/root/.kaggle/kaggle.json").write_bytes(uploaded["kaggle.json"])
        !chmod 600 /root/.kaggle/kaggle.json

!kaggle datasets download -d {DATASET_SLUG} -p {DATA_DIR} --force

In [ ]:
# Descomprimir archivos descargados y localizar CSV.
for zip_path in DATA_DIR.glob("*.zip"):
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(DATA_DIR)

csv_files = sorted(DATA_DIR.glob("*.csv"))
csv_files

In [ ]:
# Ajusta esta ruta si subiste el CSV manualmente.
CSV_PATH = csv_files[0] if csv_files else Path("/content/data/social_media_user_activity.csv")
CSV_PATH

## 5. Carga e inspeccion inicial

In [ ]:
df_raw = load_dataset(CSV_PATH)
print(f"Filas: {df_raw.shape[0]:,}")
print(f"Columnas: {df_raw.shape[1]:,}")
df_raw.head()

In [ ]:
column_summary = summarize_columns(df_raw)
column_summary

In [ ]:
report_missing_values(df_raw).head(20)

In [ ]:
report_duplicates(df_raw)

In [ ]:
split_column_types(df_raw)

## 6. EDA rapido de la variable objetivo

In [ ]:
if TARGET_COLUMN in df_raw.columns:
    display(df_raw[TARGET_COLUMN].describe())
    plt.figure(figsize=(8, 4))
    sns.histplot(df_raw[TARGET_COLUMN], bins=30, kde=True)
    plt.title(f"Distribucion de {TARGET_COLUMN}")
    plt.show()
else:
    print(f"No se encontro la columna objetivo: {TARGET_COLUMN}")

## 7. Seleccion manual de columnas

La limpieza fina la haremos juntos. Esta celda propone columnas segun el documento del proyecto, pero la lista es editable.

Regla acordada: las columnas numericas se separan en maximo 10 intervalos por cuantiles antes de crear la matriz binaria.

In [ ]:
suggested_columns = default_analysis_columns(df_raw)
suggested_columns

In [ ]:
# Editar esta lista cuando decidamos que columnas conservar.
selected_columns = suggested_columns.copy()

if TARGET_COLUMN not in selected_columns and TARGET_COLUMN in df_raw.columns:
    selected_columns.append(TARGET_COLUMN)

selected_columns

## 8. Limpieza basica y muestra de trabajo

Para iterar rapido se puede trabajar con una muestra. Cuando el flujo este validado, cambia `USE_SAMPLE = False`.

In [ ]:
df_selected = df_raw[selected_columns].copy()
df_selected = clean_selected_data(
    df_selected,
    numeric_strategy="median",
    categorical_strategy="unknown",
)
df_work = sample_dataframe(
    df_selected,
    use_sample=USE_SAMPLE,
    sample_size=SAMPLE_SIZE,
    random_state=RANDOM_STATE,
)

print(f"Dataset seleccionado: {df_selected.shape}")
print(f"Dataset de trabajo: {df_work.shape}")
df_work.head()

## 9. Control opcional de categorias raras

Si una columna categorica tiene demasiados valores distintos, puede crear una matriz binaria demasiado ancha. Esta celda agrupa categorias raras con frecuencia menor a 0.5%. Puedes desactivarla si no hace falta.

In [ ]:
GROUP_RARE_CATEGORIES = True
MIN_CATEGORY_FREQUENCY = 0.005

if GROUP_RARE_CATEGORIES:
    categorical_columns = split_column_types(df_work)["categorical"]
    df_work = reduce_rare_categories(
        df_work,
        categorical_columns=categorical_columns,
        min_frequency=MIN_CATEGORY_FREQUENCY,
    )

summarize_columns(df_work)

## 10. Discretizacion por cuantiles

Todas las columnas numericas seleccionadas se transforman a maximo `MAX_BINS` intervalos. Si una variable tiene pocos valores unicos, se generan menos intervalos.

In [ ]:
df_discretized = discretize_numeric_columns(df_work, max_bins=MAX_BINS)
df_discretized.head()

In [ ]:
if TARGET_COLUMN in df_discretized.columns:
    df_discretized[TARGET_COLUMN].value_counts(dropna=False).sort_index()

## 11. Matriz binaria

In [ ]:
binary_matrix = create_binary_matrix(df_discretized, sparse=False)
validate_binary_matrix(binary_matrix)

## 12. Itemsets frecuentes con FP-Growth

Si salen muy pocos itemsets, baja `MIN_SUPPORT`. Si salen demasiados, subelo.

In [ ]:
frequent_itemsets = mine_frequent_itemsets(binary_matrix, min_support=MIN_SUPPORT)
frequent_itemsets = frequent_itemsets.sort_values("support", ascending=False).reset_index(drop=True)
print(f"Itemsets frecuentes: {len(frequent_itemsets):,}")
frequent_itemsets.head(20)

## 13. Reglas de asociacion

In [ ]:
rules = generate_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=MIN_CONFIDENCE,
)

if not rules.empty:
    rules = rules[rules["lift"] >= MIN_LIFT].reset_index(drop=True)

print(f"Reglas generadas: {len(rules):,}")
rules_to_readable(rules).head(20)

## 14. Reglas relacionadas con felicidad autorreportada

In [ ]:
happiness_rules = filter_happiness_rules(rules, target_column=TARGET_COLUMN)
happiness_rules_readable = rules_to_readable(happiness_rules)
print(f"Reglas relacionadas con {TARGET_COLUMN}: {len(happiness_rules_readable):,}")
happiness_rules_readable.head(30)

In [ ]:
# Reglas donde felicidad aparece como consecuente.
happiness_as_consequent = filter_happiness_rules(
    rules,
    target_column=TARGET_COLUMN,
    consequent_only=True,
)
rules_to_readable(happiness_as_consequent).head(30)

## 15. Rankings de reglas

Interpretacion rapida:

- `support`: proporcion de registros donde aparece la regla.
- `confidence`: probabilidad del consecuente cuando aparecen los antecedentes.
- `lift`: fuerza de asociacion comparada con la ocurrencia esperada por azar. Valores mayores a 1 indican asociacion positiva.

In [ ]:
display_cols = ["antecedents", "consequents", "support", "confidence", "lift"]

top_by_lift = rules_to_readable(happiness_rules.sort_values("lift", ascending=False)).loc[:, display_cols].head(15)
top_by_confidence = rules_to_readable(happiness_rules.sort_values("confidence", ascending=False)).loc[:, display_cols].head(15)

print("Top reglas por lift")
display(top_by_lift)

print("Top reglas por confidence")
display(top_by_confidence)

## 16. Exportacion de resultados

In [ ]:
OUTPUT_DIR = Path("/content/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

rules_to_readable(rules).to_csv(OUTPUT_DIR / "association_rules_all.csv", index=False)
happiness_rules_readable.to_csv(OUTPUT_DIR / "association_rules_happiness.csv", index=False)
frequent_itemsets.to_csv(OUTPUT_DIR / "frequent_itemsets.csv", index=False)

list(OUTPUT_DIR.glob("*.csv"))

## 17. Conclusiones preliminares

Completar despues de revisar las reglas obtenidas:

- Reglas mas relevantes asociadas a felicidad alta/media/baja.
- Variables que aparecen con mayor frecuencia en reglas utiles.
- Limitaciones del analisis.
- Recordatorio: las reglas muestran asociaciones frecuentes, no causalidad.